# Train the multimodal dementia model on Colab

**Workflow:** train here (GPU) → download `best_model.pt` → run inference locally.

First: **Runtime → Change runtime type → GPU (T4 is fine)**.

Sections: (1) get code, (2) install, (3) smoke test, (4) real PROCESS data, (5) train, (6) download `.pt`.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/VeNoM-hubs/Neuralyzer.git"
!git clone $REPO_URL
%cd Neuralyzer
# If already cloned, pull latest instead:
# %cd Neuralyzer
# !git pull

## 2. Install dependencies
Colab already ships CUDA `torch`/`torchaudio`, so we install everything else.

In [ ]:
!pip install -q "transformers>=4.40,<5" scikit-learn>=1.3.0 PyYAML>=6.0 tqdm>=4.65.0 soundfile>=0.12.1 kaggle

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 3. Smoke test (mock data)
Confirms the full pipeline runs and loss decreases. Downloads HuBERT + BERT on first run.

In [ ]:
!python scripts/smoke_test.py

## 4. Real data — PROCESS Challenge (open Kaggle re-upload)

English speech, 3 tasks per participant; we use **CTD (Cookie Theft)** to match the paper's picture-description paradigm. Labels from the CSV, mapped **HC → 0, MCI + Dementia → 1**. Transcripts already ship with the data — **no Whisper needed**.

> **License:** derived from PROCESS/DementiaBank data. Personal reproduction/learning only.

**Kaggle token:** kaggle.com → Settings → *Create New API Token* downloads `kaggle.json`.

In [ ]:
# Robust Kaggle auth: set env vars from kaggle.json (avoids file-path/permission issues)
from google.colab import files
import json, os
up = files.upload()  # choose kaggle.json
creds = json.loads(next(iter(up.values())).decode())
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY'] = creds['key']
!kaggle datasets list -s dementia --max-size 1 >/dev/null 2>&1 && echo 'Kaggle auth OK' || echo 'AUTH FAILED'

In [ ]:
# Download + unzip (skips if already present)
!kaggle datasets download -d tahouramorovati/dementia-detection-using-speech -p /content/pitt --unzip

# Sanity: count labeled records + CTD audio
import glob
print('record folders:', len(glob.glob('/content/pitt/Archive/Process-rec-*')))
print('CTD wavs:', len(glob.glob('/content/pitt/Archive/**/*__CTD.wav', recursive=True)))

## 5. Train

**Quick sanity pass** (1 seed, 3 epochs) — do this first:
```
!python train.py --dataset process --data-root /content/pitt --single-seed --max-epochs 3
```
**Full run** (5 seeds, optimized schedule):
```
!python train.py --dataset process --data-root /content/pitt
```
Add `--tasks CTD,SFT,PFT` to use all three tasks (3× samples). OOM? lower `train.batch_size` in the YAML.

In [ ]:
!python train.py --dataset process --data-root /content/pitt --single-seed --max-epochs 3

## 6. Download the trained checkpoint
`train.py` writes `runs/best_model.pt` (weights + config bundled). Download it for local inference.

In [ ]:
from google.colab import files
import os
ckpt = 'runs/best_model.pt'
assert os.path.exists(ckpt), f'{ckpt} not found — did training finish?'
print('size (MB):', round(os.path.getsize(ckpt) / 1e6, 1))
files.download(ckpt)